In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col
import mlflow
import mlflow.spark

# %%
spark = SparkSession.builder \
    .appName("SECOP_MLflow") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/14 21:22:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/14 21:22:33 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [2]:
mlflow.set_tracking_uri("http://mlflow:5000")

In [3]:
# Crear/seleccionar experimento
experiment_name = "/SECOP_Contratos_Prediccion"
mlflow.set_experiment(experiment_name)

print(f"MLflow Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Experimento: {experiment_name}")

2026/02/14 21:29:55 INFO mlflow.tracking.fluent: Experiment with name '/SECOP_Contratos_Prediccion' does not exist. Creating a new experiment.


MLflow Tracking URI: http://mlflow:5000
Experimento: /SECOP_Contratos_Prediccion


Respuesta:
Es importante porque centraliza y versiona experimentos y artefactos en un solo lugar, facilita colaboración y comparación reproducible, y evita perder resultados en archivos locales

In [12]:
# Cargar datos
df = spark.read.parquet("/opt/spark-data/processed/secop_features.parquet")

df = df.withColumnRenamed("features_raw", "features") \
       .filter(col("label").isNotNull())

train, test = df.randomSplit([0.8, 0.2], seed=42)

print(f"Train: {train.count():,}")
print(f"Test: {test.count():,}")

# %%
evaluator = RegressionEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="rmse"
)

Train: 79,954
Test: 20,046


In [13]:
# ## RETO 2: Registrar un Experimento Baseline
with mlflow.start_run(run_name="baseline_no_regularization"):
    # Hiperparámetros
    reg_param = 0.0
    elastic_param = 0.0
    max_iter = 100

    # Log de hiperparámetros
    mlflow.log_param("regParam", reg_param)
    mlflow.log_param("elasticNetParam", elastic_param)
    mlflow.log_param("maxIter", max_iter)
    mlflow.log_param("model_type", "LinearRegression")

    # Entrenar modelo
    lr = LinearRegression(
        featuresCol="features",
        labelCol="label",
        predictionCol="prediction",
        regParam=reg_param,
        elasticNetParam=elastic_param,
        maxIter=max_iter
    )
    model = lr.fit(train)

    # Evaluar (RMSE)
    predictions = model.transform(test)
    rmse = evaluator.evaluate(predictions)

    # Log de métricas
    mlflow.log_metric("rmse", float(rmse))

    # Guardar modelo
    mlflow.spark.log_model(model, "model")

    print(f"✅ Run registrado en MLflow | RMSE: ${rmse:,.2f}")


26/02/14 21:49:38 WARN Instrumentation: [439fc0b3] regParam is zero, which might cause numerical instability and overfitting.
26/02/14 21:49:40 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/02/14 21:49:40 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.VectorBLAS
26/02/14 21:49:40 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK
26/02/14 21:49:40 WARN Instrumentation: [439fc0b3] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
2026/02/14 21:49:57 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpfy_5m51x/model, flavor: spark), fall back to return ['pyspark==3.5.0']. Set logging level to DEBUG to see the full traceback.
/usr/local/lib/python3.11/site-packages/_distutils_hack/__init__.py:30: UserWarning: Setuptools is replacing distutils. Support for repla

✅ Run registrado en MLflow | RMSE: $3,030,748,234.06


In [16]:
# ## RETO 3: Registrar Múltiples Experimentos
from pyspark.ml.evaluation import RegressionEvaluator

# Evaluadores para 3 métricas
evaluator_rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
evaluator_mae  = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae")
evaluator_r2   = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")

experiments = [
    {"name": "ridge_l2",   "reg": 0.1, "elastic": 0.0, "type": "Ridge"},
    {"name": "lasso_l1",   "reg": 0.1, "elastic": 1.0, "type": "Lasso"},
    {"name": "elasticnet", "reg": 0.1, "elastic": 0.5, "type": "ElasticNet"},
]

max_iter = 100

for exp in experiments:
    with mlflow.start_run(run_name=exp["name"]):
        # Log parámetros
        mlflow.log_param("regParam", exp["reg"])
        mlflow.log_param("elasticNetParam", exp["elastic"])
        mlflow.log_param("maxIter", max_iter)
        mlflow.log_param("model_type", "LinearRegression")
        mlflow.log_param("regularization_type", exp["type"])

        # Entrenar modelo
        lr = LinearRegression(
            featuresCol="features",  # ✅ aquí el cambio
            labelCol="label",
            predictionCol="prediction",
            regParam=exp["reg"],
            elasticNetParam=exp["elastic"],
            maxIter=max_iter
        )
        model = lr.fit(train)

        # Evaluar
        preds = model.transform(test)
        rmse = evaluator_rmse.evaluate(preds)
        mae  = evaluator_mae.evaluate(preds)
        r2   = evaluator_r2.evaluate(preds)

        # Log métricas
        mlflow.log_metric("rmse", float(rmse))
        mlflow.log_metric("mae", float(mae))
        mlflow.log_metric("r2", float(r2))

        # Guardar modelo
        mlflow.spark.log_model(model, "model")

        print(f"✅ {exp['name']} | RMSE=${rmse:,.2f} | MAE=${mae:,.2f} | R2={r2:.4f}")



2026/02/14 21:54:17 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpzl92nhsm/model, flavor: spark), fall back to return ['pyspark==3.5.0']. Set logging level to DEBUG to see the full traceback.


✅ ridge_l2 | RMSE=$3,030,754,693.01 | MAE=$152,105,378.77 | R2=0.0029


2026/02/14 21:54:33 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpjrb19l2s/model, flavor: spark), fall back to return ['pyspark==3.5.0']. Set logging level to DEBUG to see the full traceback.


✅ lasso_l1 | RMSE=$3,030,748,798.68 | MAE=$152,110,708.07 | R2=0.0029


2026/02/14 21:54:48 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmp304jqk0s/model, flavor: spark), fall back to return ['pyspark==3.5.0']. Set logging level to DEBUG to see the full traceback.


✅ elasticnet | RMSE=$3,030,748,798.69 | MAE=$152,110,708.62 | R2=0.0029


¿Por qué registrar múltiples métricas y no solo RMSE?
Porque RMSE castiga más los errores grandes (sensible a outliers), MAE mide el error promedio de forma más estable y R² indica cuánto explica el modelo de la variación del objetivo. Con una sola métrica podrías escoger un modelo que “se ve bien” en RMSE pero es peor en MAE o explica poco (R² bajo).

# ## RETO 4: Explorar MLflow UI
Mejor modelo en MLflow UI: lasso_l1
RMSE del mejor modelo: 3,030,748,798.68

Observaciones:
- Lasso (L1) obtuvo el menor RMSE, pero la diferencia frente a ElasticNet es mínima; Ridge quedó ligeramente por encima en RMSE.
- No hay una correlación fuerte o clara entre “más regularización” y mejor rendimiento, porque Ridge/Lasso/ElasticNet quedaron muy cercanos.
- El R² (~0.0029) es muy bajo en todos los runs: el modelo explica muy poca variación del objetivo, así que conviene revisar features/transformaciones (p. ej. log(label), más variables, interacciones, o modelos no lineales).
- Para compartir con el equipo: enviar el link del experimento en MLflow + el Run ID del mejor run y adjuntar un reporte de métricas como artifact (txt) para reproducibilidad.


In [17]:
# ## RETO 5: Agregar Artefactos Personalizados
import os
import matplotlib.pyplot as plt
from pyspark.ml.evaluation import RegressionEvaluator

# Evaluadores
evaluator_rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
evaluator_mae  = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae")
evaluator_r2   = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")

with mlflow.start_run(run_name="model_with_artifacts"):
    # Mejor configuración (según tu UI: Lasso)
    reg_param = 0.1
    elastic_param = 1.0
    max_iter = 100

    # Log parámetros
    mlflow.log_param("regParam", reg_param)
    mlflow.log_param("elasticNetParam", elastic_param)
    mlflow.log_param("maxIter", max_iter)
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_param("regularization_type", "Lasso")

    # Entrenar modelo
    lr = LinearRegression(
        featuresCol="features",
        labelCol="label",
        predictionCol="prediction",
        regParam=reg_param,
        elasticNetParam=elastic_param,
        maxIter=max_iter
    )
    model = lr.fit(train)

    # Predicciones y métricas
    preds = model.transform(test)
    rmse = evaluator_rmse.evaluate(preds)
    mae  = evaluator_mae.evaluate(preds)
    r2   = evaluator_r2.evaluate(preds)

    mlflow.log_metric("rmse", float(rmse))
    mlflow.log_metric("mae", float(mae))
    mlflow.log_metric("r2", float(r2))

    # 1) Reporte de texto como artefacto
    report = f"""
REPORTE DE MODELO
==================
Run: model_with_artifacts
Modelo: LinearRegression (Lasso)

PARAMETROS
----------
regParam: {reg_param}
elasticNetParam: {elastic_param}
maxIter: {max_iter}

METRICAS
--------
RMSE: {rmse:,.2f}
MAE:  {mae:,.2f}
R2:   {r2:.4f}
""".strip()

    mlflow.log_text(report, "model_report.txt")

    # 2) Bonus: gráfico Predicho vs Real (muestra para no reventar memoria)
    sample = preds.select("label", "prediction").dropna().limit(5000).toPandas()

    plt.figure()
    plt.scatter(sample["label"], sample["prediction"])
    plt.xlabel("Real (label)")
    plt.ylabel("Predicho (prediction)")

    plot_path = "/tmp/predictions_vs_real.png"
    plt.savefig(plot_path)
    plt.close()

    mlflow.log_artifact(plot_path)

    # 3) Guardar modelo como artefacto
    mlflow.spark.log_model(model, "model")

    print(f"✅ Artefactos guardados: model_report.txt y predictions_vs_real.png")
    print(f"✅ Métricas: RMSE={rmse:,.2f} | MAE={mae:,.2f} | R2={r2:.4f}")


2026/02/14 22:00:53 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmppv5y_oam/model, flavor: spark), fall back to return ['pyspark==3.5.0']. Set logging level to DEBUG to see the full traceback.


✅ Artefactos guardados: model_report.txt y predictions_vs_real.png
✅ Métricas: RMSE=3,030,748,798.68 | MAE=152,110,708.07 | R2=0.0029


# Preguntas de Reflexión
¿Qué ventajas tiene MLflow sobre guardar métricas en archivos CSV?
Respuesta: MLflow centraliza en un solo lugar parámetros, métricas, modelos y artefactos por cada run, con trazabilidad y comparación en UI. A diferencia de un CSV, no pierdes el contexto (qué hiperparámetros se usaron, qué versión de modelo, qué artefactos quedaron), puedes filtrar/ordenar runs fácilmente y mantener historial reproducible para auditoría y seguimiento.

¿Cómo implementarías MLflow en un proyecto de equipo?
Respuesta: Montaría un tracking server compartido (Docker/K8s) con almacenamiento de artefactos centralizado (volumen compartido o S3/MinIO). Definiría convenciones: nombre de experimento, naming de runs, y métricas mínimas obligatorias (rmse/mae/r2). Integraría el log en los pipelines de entrenamiento y haría que todos registren runs en el mismo experimento para comparar y seleccionar el mejor, compartiendo links y Run IDs.

¿Qué artefactos adicionales guardarías además del modelo?
Respuesta: Guardaría reportes de métricas (como model_report.txt), gráficos (real vs predicho, residuales), muestra de predicciones (top errores), configuración del pipeline de features (VectorAssembler/PCA), “model card” con supuestos/limitaciones, y referencia/versión del dataset usado (ruta, fecha, filtros) para reproducibilidad.

¿Cómo automatizarías el registro de experimentos?
Respuesta: Crearía una función/pipeline que reciba una lista de configuraciones (grid de hiperparámetros) y ejecute entrenar→evaluar→log automático en MLflow. Luego lo correría en un orquestador (Airflow/Prefect) o CI/CD (GitHub Actions), guardando el mejor run según criterio (ej. menor RMSE y MAE, con R² aceptable) y registrando automáticamente el modelo ganador en el Model Registry.

In [18]:
print("\n" + "="*60)
print("RESUMEN MLFLOW TRACKING")
print("="*60)
print("Verifica que hayas completado:")
print("  [ ] Configurado MLflow tracking server")
print("  [ ] Registrado experimento baseline")
print("  [ ] Registrado al menos 3 experimentos adicionales")
print("  [ ] Explorado MLflow UI")
print("  [ ] Comparado métricas entre runs")
print(f"  [ ] Accede a MLflow UI: http://localhost:5000")
print("="*60)


RESUMEN MLFLOW TRACKING
Verifica que hayas completado:
  [ ] Configurado MLflow tracking server
  [ ] Registrado experimento baseline
  [ ] Registrado al menos 3 experimentos adicionales
  [ ] Explorado MLflow UI
  [ ] Comparado métricas entre runs
  [ ] Accede a MLflow UI: http://localhost:5000


In [19]:
spark.stop()